# Coding Homework 06: Projection and Positive-Definite Gram Matrices

This assignment uses projection to separate represented information from an orthogonal residual, then connects Gram matrices to positive definiteness and stable computation. You will also add a feature-subspace diagnostic to your cumulative mini-transformer.

**Estimated time:** 2–3 hours
**Submit:** this notebook and `mini_transformer.py`


## Start here: how to work through this notebook

You do not need to remember an earlier programming course. Work one small step
at a time. The mathematics is the goal; the programming examples are here to
help you express it.

1. Save your own copy of the notebook. A **cell** is one block of text or code.
   Click a code cell and press **Shift+Enter** to run it. Run cells from the top
   down: later cells use names created by earlier cells.
2. Run the setup cell first. `import numpy as np` makes NumPy's array tools
   available under the short name `np`.
3. Read the explanation and run the small practice example before attempting
   the corresponding task. Practice cells are supplied; you may experiment
   with them, but restore them before the final run.
4. Edit only the requested task or response. `#` begins a comment. `pass` is a
   placeholder that does nothing; a function ending at `pass` returns `None`
   (no result). Replace it with your calculation and a `return` statement.
5. Running a cell containing `def` stores the function; it does not call it.
   After editing the definition, run that cell again, then run its check cell.
   A successful `assert` may print nothing. A skipped check is not a pass:
   if a result is still `None`, the task is unfinished.
6. If a check fails, stop there. Read the last line of the error, print the
   relevant array's `.shape`, and fix this task before running cells that need it.
   Errors below an unfinished task are expected and are not separate failures.
7. Save, restart the kernel/runtime (the Python session), and run everything
   from the beginning once your work is complete. This catches accidental use
   of a value left over from an earlier attempt.

**Reading code:** `=` stores a value; `==` compares values; `@` multiplies
matrices; `*` multiplies entries or scales an array; `**2` squares entries.
Python indices start at **0**: mathematical entry $a_{1,1}$ is `A[0, 0]`.
`A.shape` is `(number_of_rows, number_of_columns)`, `A.ndim` counts dimensions,
and `A.T` transposes a matrix. An ordinary vector has shape `(n,)`.

**If you get stuck:** record the function name, the smallest input that fails,
its shape, the expected mathematical result, and the last line of the error.
Include these when asking for help. You do not need to rewrite the whole cell.

**Plan your time:** if Python is new to you, allow extra time beyond the
headline estimate and work in several short sessions. Start with setup and
the practice example; then finish one task at a time.

More worked examples and an error-message checklist: [Python and NumPy help](`mt_utils.py` / `mini_transformer.py` in this folder).


### Which file do I edit?

Download the notebook and `mini_transformer.py` from this assignment's
`student` folder. Keep both in the same working folder. Use the supplied
completed starting model; you do not need to recover earlier submissions.

- **Notebook:** fill the named notebook functions and written responses.
- **`mini_transformer.py`:** open this plain-text Python file in an editor,
  find the named milestone function, replace its placeholder, and save.
  Keep the other supplied model functions. A `.py` file cannot see a function
  merely because you defined it in the notebook.
- **Colab:** use File → Upload notebook; in the Files panel upload the `.py`
  file. Double-click that file to edit it and save. If it is not editable in
  your interface, edit it locally and upload the replacement. Download both
  edited files before ending the session; Colab uploads are temporary.
- **Local Jupyter:** open the notebook from the folder containing both files.
  If setup reports `ModuleNotFoundError: mini_transformer`, check that folder
  and the spelling before trying to install anything.

Python remembers imported files. After saving a module edit, use the reload
cell beside the milestone (or restart and run from the top). Submit **both**
files to the course portal. Wait for the results and inspect the feedback; upload
alone does not confirm the work passed.


In [1]:
# Supplied practice: run this unchanged, then trace the result by hand.
import numpy as np

def double_and_total(values):
    values = np.asarray(values, dtype=float)  # turn a list into a numerical array
    doubled = 2 * values                     # one intermediate calculation
    total = np.sum(doubled)                   # add all entries
    return doubled, total                    # send two results back to the caller

practice_vector, practice_total = double_and_total([1, -2, 3])
print(practice_vector)        # [ 2. -4.  6.]
print(practice_vector.shape)  # (3,)
print(practice_total)         # 4.0
assert practice_total == 4.0


[ 2. -4.  6.]
(3,)
4.0


## 0. Setup and terminology

Keep `mini_transformer.py` beside this notebook. A **token activation** is simply one row of numbers produced for a token inside the model. A **feature subspace** is a lower-dimensional collection of feature directions. Projecting an activation onto that subspace asks which part can be represented by those directions.


In [2]:
import numpy as np
import mini_transformer as mt

np.set_printoptions(precision=5, suppress=True)
W_check = mt.orthogonal_matrix(4, 4, seed=412)
assert np.allclose(W_check.T @ W_check, np.eye(4), atol=1e-10)
print("Setup complete; the cumulative HW05 model is available.")


Setup complete; the cumulative HW05 model is available.


## 1. Projection onto a column space

Suppose the independent columns of $A$ span a subspace of $\mathbb{R}^m$. Reduced QR gives $A=QR$, where the columns of $Q$ are an orthonormal basis for the same subspace. The orthogonal projection matrix is

$$P=QQ^T.$$

It is symmetric ($P^T=P$) and idempotent ($P^2=P$). Use QR rather than explicitly forming $(A^TA)^{-1}$; it reuses prior course ideas and is numerically better behaved.

A projection onto Col(A) keeps vectors in that column space and sends perpendicular vectors to zero. Symmetry and idempotence alone do not identify the subspace: the identity matrix also has both properties, but it keeps every direction.


### Your task: `projection_matrix` — build it in four steps

**Input:** `A` has shape `(m, k)` and independent columns. **Output:** a matrix
of shape `(m, m)`; it acts on vectors with `m` entries.

1. Convert with `np.asarray(A, dtype=float)`. Require `A.ndim == 2` before
   reading both shape entries; reject empty dimensions with `ValueError`.
2. Count independent columns with `np.linalg.matrix_rank(A, tol=tol)`.
   Compare that number with `A.shape[1]`; reject a mismatch with `ValueError`.
3. Call `np.linalg.qr(A, mode="reduced")`. It returns **two** arrays; assign
   them with `Q, R = ...`. You only need `Q` for the next step.
4. Translate the displayed projection formula using `@` and `.T`, and
   **return** the resulting matrix. Printing it alone will not return it.

`tol=1e-10` means $10^{-10}$, a small tolerance for roundoff. Keep this
argument and pass it to the rank check. For a `(3, 2)` input, inspect
`Q.shape`: it should be `(3, 2)`, so the final product is `(3, 3)`.

**Check meaning:** symmetry and idempotence are necessary; the later checks
also verify that columns of `A` are kept and perpendicular directions vanish.
An identity matrix would fail that last requirement.


In [3]:
def projection_matrix(A, tol=1e-10):
    """Return the orthogonal projector onto Col(A).

    A must be a nonempty matrix with linearly independent columns.
    """
    # 1. Convert to array and validate dimensions
    A = np.asarray(A, dtype=float)
    if A.ndim != 2:
        raise ValueError("A must be a 2D matrix.")

    # 2. Count independent columns and compare with column count
    rank = np.linalg.matrix_rank(A, tol=tol)
    if rank != A.shape[1]:
        raise ValueError("Columns of A must be linearly independent.")

    # 3. Compute reduced QR factorization
    Q, R = np.linalg.qr(A, mode="reduced")

    # 4. Return the projection matrix Q @ Q.T
    return Q @ Q.T


In [4]:
A_proj = np.array([[1., 0.], [1., 1.], [0., 1.]])
P = projection_matrix(A_proj)
assert P.shape == (3, 3)
assert np.allclose(P.T, P, atol=1e-10)
assert np.allclose(P @ P, P, atol=1e-10)
assert np.allclose(P @ A_proj, A_proj, atol=1e-10)
# A vector perpendicular to every column must project to zero.
n = np.array([1., -1., 1.])
assert np.allclose(A_proj.T @ n, 0.0, atol=1e-10)
assert np.allclose(P @ n, 0.0, atol=1e-10)
print("Projection matrix:\n", P)


Projection matrix:
 [[ 0.66667  0.33333 -0.33333]
 [ 0.33333  0.66667  0.33333]
 [-0.33333  0.33333  0.66667]]


## 2. Represented component and lost component

For a vector $b$, the projection $p=Pb$ is the closest vector in $\operatorname{Col}(A)$. The residual $r=b-p$ is orthogonal to every column of $A$, so $A^Tr=0$. In a compression setting, $p$ is the part retained by the chosen subspace and $r$ is the part it cannot represent.


### Your task: `project_vector` — reuse the function you just wrote

**Input:** `A` is `(m, k)`; `b` is `(m,)`. **Output:** two vectors, each `(m,)`.

1. Convert `A` and `b` to floating-point arrays. Require a matrix `A` and a
   one-dimensional `b` with one entry per row of `A`.
2. Call `projection_matrix(A, tol=tol)` and store the result as `P`.
3. Compute the represented part `p` from the matrix-vector product in the
   formula above. Compute a separate residual variable by subtraction.
4. Return the two vectors in the order **projection, residual**. The statement
   `p, residual = project_vector(...)` unpacks them, like the warm-up example.

If a check says `NoneType`, first look for a missing `return`. If it reports a
matrix-multiplication dimension mismatch, print `P.shape` and `b.shape`.
The residual should be perpendicular to the represented subspace, not
necessarily the zero vector.


In [5]:
def project_vector(A, b, tol=1e-10):
    """Return (projection, orthogonal residual) for b onto Col(A)."""
    # 1. Convert A and b to floating-point arrays and validate shapes
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if A.ndim != 2:
        raise ValueError("A must be a 2D matrix.")
    if b.ndim != 1 or len(b) != A.shape[0]:
        raise ValueError("b must be a 1D vector with length equal to the number of rows of A.")

    # 2. Get the projection matrix P
    P = projection_matrix(A, tol=tol)

    # 3. Compute represented part p and orthogonal residual
    p = P @ b
    residual = b - p

    # 4. Return projection and residual
    return p, residual


In [6]:
b_proj = np.array([1., 2., 4.])
p, residual = project_vector(A_proj, b_proj)
assert np.allclose(A_proj.T @ residual, 0.0, atol=1e-10)
assert np.allclose(p + residual, b_proj, atol=1e-10)
assert np.isclose(np.dot(p, residual), 0.0, atol=1e-10)
print("projected component:", p)
print("orthogonal residual:", residual)


projected component: [-0.  3.  3.]
orthogonal residual: [ 1. -1.  1.]


## 3. Positive definite and positive semidefinite matrices

For a real symmetric matrix $M$:

- **positive definite (PD):** $x^TMx>0$ for every nonzero $x$; all eigenvalues are positive;
- **positive semidefinite (PSD):** $x^TMx\ge 0$; eigenvalues may include zero;
- negative definite and negative semidefinite use the reversed inequalities;
- **indefinite:** the quadratic form takes both positive and negative values.

Complete the classifier below. A nonsymmetric matrix receives its own label because these eigenvalue tests are being used only for real symmetric matrices.


### Your task: `classify_definiteness` — translate a decision table

**Input:** a square real matrix `(n, n)`. **Output:** one exact text label
shown in the check cell. Convert to an array; reject a nonsquare shape.
Use `np.allclose(M, M.T, atol=tol, rtol=0)` to test symmetry. Return
`"not symmetric"` before using `np.linalg.eigvalsh` if that test fails.

For a symmetric matrix, store the eigenvalues and apply this table **in order**:

| Test on all eigenvalues | Return text |
|---|---|
| Every value is greater than `tol` | `"positive definite"` |
| Every value is at least `-tol` | `"positive semidefinite"` |
| Every value is less than `-tol` | `"negative definite"` |
| Every value is at most `tol` | `"negative semidefinite"` |
| Otherwise | `"indefinite"` |

Use `np.all(...)` to turn an array of comparisons into one Boolean. For
example, `np.all(np.array([2., 3.]) > 1)` is `True`. Use `if`/`elif`/`else`
and indent each branch by four spaces. Test definite cases before their
corresponding semidefinite cases. The zero matrix satisfies both semidefinite
definitions; this assignment labels it **positive semidefinite** by the table's
order. Tiny values within the tolerance are treated as zero.


In [7]:
def classify_definiteness(M, tol=1e-10):
    """Classify a real symmetric matrix; label nonsymmetric inputs separately."""
    # 1. Convert to array and validate square shape
    M = np.asarray(M, dtype=float)
    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        raise ValueError("M must be a square 2D matrix.")

    # 2. Test for symmetry
    if not np.allclose(M, M.T, atol=tol, rtol=0):
        return "not symmetric"

    # 3. Compute eigenvalues
    eigenvalues = np.linalg.eigvalsh(M)

    # 4. Classify based on the ordered decision table
    if np.all(eigenvalues > tol):
        return "positive definite"
    elif np.all(eigenvalues >= -tol):
        return "positive semidefinite"
    elif np.all(eigenvalues < -tol):
        return "negative definite"
    elif np.all(eigenvalues <= tol):
        return "negative semidefinite"
    else:
        return "indefinite"


In [8]:
definiteness_examples = {
    "positive definite": np.array([[3., 1.], [1., 2.]]),
    "positive semidefinite": np.array([[1., 1.], [1., 1.]]),
    "negative definite": np.array([[-3., -1.], [-1., -2.]]),
    "negative semidefinite": np.array([[-1., -1.], [-1., -1.]]),
    "indefinite": np.array([[1., 0.], [0., -1.]]),
    "not symmetric": np.array([[1., 2.], [0., 1.]]),
}
for expected_label, matrix in definiteness_examples.items():
    actual_label = classify_definiteness(matrix)
    assert actual_label == expected_label, (expected_label, actual_label)
print("All six definiteness labels passed.")


All six definiteness labels passed.


## 4. Gram matrices and ridge regularization

For any matrix $A$, its column **Gram matrix** $G=A^TA$ stores all column dot products. It is always PSD because $x^TGx=\lVert Ax\rVert^2\ge0$. It is PD exactly when the columns of $A$ are independent. If they are dependent, adding a **ridge term** $\alpha I$ with $\alpha>0$ shifts every eigenvalue upward and makes the matrix PD:

$$G_\alpha=A^TA+\alpha I.$$

A PD matrix has a Cholesky factorization $G_\alpha=LL^T$.


### Your task: `regularized_gram` — choose the identity's size

**Input:** `A` is `(m, k)` and `ridge` is one positive finite number.
**Output:** `(k, k)`. A **scalar** means one number, not a vector or matrix.

1. Convert `A` to an array and require two dimensions and at least one column.
2. Require `np.isscalar(ridge)`, `np.isfinite(ridge)`, and `ridge > 0`.
   Raise `ValueError` otherwise. `np.isfinite` rejects infinity and NaN
   (“not a number,” such as an undefined calculation).
3. Compute the column Gram matrix. Check its shape before continuing.
4. `np.eye(k)` creates a `k`-by-`k` identity. Scale that identity by `ridge`,
   add it to the Gram matrix, and return the sum.

For a `(3, 2)` input you need a `(2, 2)` identity, not a `(3, 3)` one.
The Cholesky factorization below is supplied: run it to check that the positive
ridge shift made this example positive definite.


In [9]:
def regularized_gram(A, ridge):
    """Return A.T @ A + ridge * I for a positive ridge value."""
    # 1. Convert to array and validate dimensions
    A = np.asarray(A, dtype=float)
    if A.ndim != 2:
        raise ValueError("A must be a 2D matrix.")
    if A.shape[1] < 1:
        raise ValueError("A must have at least one column.")

    # 2. Validate ridge
    if not (np.isscalar(ridge) and np.isfinite(ridge) and ridge > 0):
        raise ValueError("ridge must be a positive finite scalar.")

    # 3. Compute Gram matrix
    G = A.T @ A

    # 4. Add the ridge times identity matrix
    k = A.shape[1]
    return G + ridge * np.eye(k)


In [10]:
A_dependent = np.array([[1., 2.], [2., 4.], [3., 6.]])
G = A_dependent.T @ A_dependent
assert classify_definiteness(G) == "positive semidefinite"
G_ridge = regularized_gram(A_dependent, ridge=0.25)
assert classify_definiteness(G_ridge) == "positive definite"
L = np.linalg.cholesky(G_ridge)
assert np.allclose(L @ L.T, G_ridge, atol=1e-10)
print("regularized Gram eigenvalues:", np.linalg.eigvalsh(G_ridge))


regularized Gram eigenvalues: [ 0.25 70.25]


## 5. Mini-transformer milestone: project token features

The forward block produces a matrix $Y$ with one token activation per row. If the columns of $Q$ are orthonormal feature directions, then

$$Y_{\text{projected}}=YQQ^T$$

keeps the part of each row in $\operatorname{Col}(Q)$. Implement `project_features` in `mini_transformer.py`. This is an **analysis/compression tool applied after the block**; it is not a replacement for attention.


### Your task in the model file: `project_features`

In `mini_transformer.py`, find `def project_features`. The supplied validation
helper already checks compatible shapes and orthonormal columns of `Q`.
Keep that helper call; replace only the placeholder below it.

| Object | Meaning | Shape |
|---|---|---|
| `X` (called `Y` below) | one activation row per token | `(n, d)` |
| `Q` | orthonormal feature directions | `(d, k)` |
| coordinates | one coordinate per retained direction | `(n, k)` |
| reconstructed rows | projected activations in original features | `(n, d)` |

1. Multiply activation rows by `Q` to get their coordinates.
2. Map those coordinates back using the transpose of `Q`.
3. Return that reconstruction. Do not call a notebook function from the module.
4. Save the `.py` file, run the reload cell, then the supplied model experiment.

The long block below is **supplied setup**: it makes toy weights and calls the
model. You need not rewrite it. Its checks say that the residual is
perpendicular to the chosen directions and that squared sizes add correctly.
“Retained energy” here is a ratio of sums of squared entries, not an accuracy
score or a physical energy.


In [11]:
import importlib
mt = importlib.reload(mt)


In [12]:
X = np.array([[1., 0., 1., 0.],
              [0., 1., 1., 0.],
              [1., 1., 0., 1.]])
W_Q, W_K, W_V = mt.orthogonal_projection_matrices(4, 4, seed=412)
rng = np.random.default_rng(412)
W1 = rng.standard_normal((4, 7)) / np.sqrt(4)
W2 = rng.standard_normal((7, 4)) / np.sqrt(7)
Y = mt.transformer_block(
    X, W_Q, W_K, W_V, W1, np.zeros(7), W2, np.zeros(4),
    mask=mt.causal_mask(len(X)),
)
Q_features = mt.orthogonal_matrix(4, 2, seed=23)
Y_projected = mt.project_features(Y, Q_features)
Y_residual = Y - Y_projected
assert Y_projected.shape == Y.shape
assert np.allclose(Y_residual @ Q_features, 0.0, atol=1e-10)
assert np.isclose(
    np.linalg.norm(Y) ** 2,
    np.linalg.norm(Y_projected) ** 2 + np.linalg.norm(Y_residual) ** 2,
    atol=1e-9,
)
print("fraction of activation energy retained:",
      np.linalg.norm(Y_projected) ** 2 / np.linalg.norm(Y) ** 2)


fraction of activation energy retained: 0.2355953182925286


## 6. Attention scores are not projection matrices

A standard attention head forms separate queries $Q=XW_Q$ and keys $K=XW_K$, then uses $QK^T/\sqrt{d}$. This score matrix is generally not symmetric, so it is not a Gram matrix and not an orthogonal projector.

If the same vectors are used on both sides, $QQ^T$ is a **tied query/key Gram matrix**. It is PSD, but it is still not generally idempotent. Softmax attention then turns scores into weights for a learned weighted average.


### Supplied comparison: run and interpret

`queries` and `Q_features` are different objects. The former are transformed
token rows used for attention; the latter are orthonormal basis columns used
for projection. Similar letters do not give them the same mathematical role.
Run the supplied comparison and record which matrix is symmetric. You are
not asked to implement attention again.


In [13]:
queries = Y @ W_Q
keys = Y @ W_K
standard_scores = mt.scaled_dot_product_scores(queries, keys)
tied_gram = queries @ queries.T
assert not np.allclose(standard_scores, standard_scores.T)
assert np.all(np.linalg.eigvalsh(tied_gram) >= -1e-10)
assert not np.allclose(tied_gram @ tied_gram, tied_gram)
print("standard scores symmetric:", np.allclose(standard_scores, standard_scores.T))
print("tied Gram classification:", classify_definiteness(tied_gram))


standard scores symmetric: False
tied Gram classification: positive definite


### Short interpretation

In 3–5 sentences, explain what the projected activation and residual mean, why a tied query/key Gram matrix is PSD, and why that does **not** make ordinary attention an orthogonal projection.


### Write your explanation

Replace the placeholder **inside** the triple quotation marks; keep those
marks and `.strip()`. Triple quotes allow several lines of text. The check
requires at least 40 words, so aim for 3–5 complete sentences. Address each
of the three questions above, referring to one check or output you observed.


In [14]:
projection_interpretation = """
The projected activation is the part of each feature vector that lies inside the
chosen subspace, while the residual is the component left over outside that
subspace.
In the output, the projection plus the residual reconstructs the original activation,
and the residual is orthogonal to the projection subspace.
A tied query/key Gram matrix is positive semidefinite because it has the form of
a matrix multiplied by its transpose, so its quadratic form cannot be negative.
However, ordinary attention is not an orthogonal projection because attention weights
come from softmaxed similarity scores and generally do not produce a symmetric,
idempotent projection matrix.
""".strip()
assert len(projection_interpretation.split()) >= 40
print(projection_interpretation)


The projected activation is the part of each feature vector that lies inside the
chosen subspace, while the residual is the component left over outside that
subspace.
In the output, the projection plus the residual reconstructs the original activation,
and the residual is orthogonal to the projection subspace.
A tied query/key Gram matrix is positive semidefinite because it has the form of
a matrix multiplied by its transpose, so its quadratic form cannot be negative.
However, ordinary attention is not an orthogonal projection because attention weights
come from softmaxed similarity scores and generally do not produce a symmetric,
idempotent projection matrix.


## 7. Submission and AI-use note

Restart the kernel, run all cells, and confirm every assertion passes. Keep this notebook alongside `mt_utils.py` / `mini_transformer.py` in the same folder when re-running.



In [15]:
ai_use_note = "No AI assistance used."  # Replace if applicable
assert ai_use_note.strip()
print(ai_use_note)


No AI assistance used.
